In [ ]:
import pandas as pd
from signals.can_parser import CANSignalParser



In [2]:
print("Loading ARXML and hunting for signals... (This will take a few minutes)")
parser = CANSignalParser("ETH_CAN.arxml")
df_signals = parser.to_dataframe()

print("\n" + "="*50)
print(" 🔍 ARXML SIGNAL NAME DISCOVERY")
print("="*50)


Loading ARXML and hunting for signals... (This will take a few minutes)
2026-04-30 19:56:46,815 - INFO - [can_parser] - Parsing CAN ARXML: ETH_CAN.arxml
2026-04-30 19:57:10,553 - INFO - [can_parser] - Filtered for 8398 CAN signals (starting with 'I').
2026-04-30 20:01:15,883 - INFO - [can_parser] - Successfully extracted 8398 CAN signals. (8 missing topology paths).

 🔍 ARXML SIGNAL NAME DISCOVERY


In [3]:
import os
from core.logger import log

In [5]:
import os
import pandas as pd
from IPython.display import display

# ==========================================
# CONFIGURATION
# ==========================================
EXCEL_FILE = "Intermediate_Requirements.xlsx"
sheet_name = "E2E_CAN"
req_signal_column = "Can Signal"

print(f"Loading existing Excel sheet: {EXCEL_FILE} (Tab: {sheet_name})...")
df_req = pd.read_excel(EXCEL_FILE, sheet_name=sheet_name)

# ==========================================
# NORMALIZE KEYS FOR MATCHING (THE FIX IS HERE)
# ==========================================
# 1. Requirements side: Lowercase and strip
df_req['match_key'] = df_req[req_signal_column].astype(str).str.strip().str.lower()

# 2. ARXML Database side: Lowercase, strip, and remove the EXACT leading 'i'
df_signals['match_key'] = df_signals['Signal_Name'].astype(str).str.strip().str.lower()
df_signals['match_key'] = df_signals['match_key'].str.replace(r'^i', '', regex=True)

# 3. Drop duplicates on the match key in the database to prevent multiplying rows
df_signals_unique = df_signals.drop_duplicates(subset=['match_key'])

# ==========================================
# PERFORM THE MERGE
# ==========================================
columns_to_add = [
    'match_key', 'Signal_String', 'Cluster', 'TX_Node', 'RX_Nodes', 
    'Periodicity_ms', 'Base_Type', 'CAPL_Type', 'Unit', 
    'Resolution', 'Offset', 'Min', 'Max'
]

# Clean up any previously appended columns so we don't get _x, _y suffixes on re-runs
existing_cols = df_req.columns.tolist()
cols_to_drop = [c for c in columns_to_add if c in existing_cols and c != 'match_key']
df_req = df_req.drop(columns=cols_to_drop)

# Merge!
df_updated = pd.merge(
    df_req, 
    df_signals_unique[columns_to_add], 
    on='match_key', 
    how='left'
)

# Flag for our statistics output
df_updated['Is_Found_In_DB'] = df_updated['Signal_String'].notna()

# Split for the visual output before we drop the temp keys
df_matched = df_updated[df_updated['Is_Found_In_DB'] == True].copy()
df_missing = df_updated[df_updated['Is_Found_In_DB'] == False].copy()

# Remove temporary match flags before saving to Excel
df_updated = df_updated.drop(columns=['match_key', 'Is_Found_In_DB'])

# ==========================================
# WRITE BACK TO EXCEL
# ==========================================
print("Saving updated CAN data back to Excel...")
try:
    with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_updated.to_excel(writer, sheet_name=sheet_name, index=False)
    print("✅ CAN Excel update completed successfully!")
except PermissionError:
    print("❌ Permission Denied: Close the Excel file and try again.")
except Exception as e:
    print(f"❌ Failed to write to Excel: {e}")

# ==========================================
# VISUALIZATION & STATISTICS
# ==========================================
total_reqs = len(df_matched) + len(df_missing)
matched_count = len(df_matched)
missing_count = len(df_missing)

print("\n=== FINAL CAN VALIDATION STATISTICS ===")
print(f"Total Requirements Processed : {total_reqs}")
print(f"✅ Matched (Normalized)       : {matched_count} ({(matched_count/total_reqs)*100 if total_reqs else 0:.2f}%)")
print(f"❌ Still Missing             : {missing_count}\n")

# Configure Pandas for beautiful notebook rendering
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

if not df_matched.empty:
    print("--- 🟢 Top 5 SUCCESSFULLY MATCHED CAN Requirements ---")
    display_cols_matched = [
        'REQ ID', 'Can Signal', 'Cluster', 'TX_Node', 'RX_Nodes', 
        'Periodicity_ms', 'Signal_String', 'CAPL_Type'
    ]
    display(df_matched[[c for c in display_cols_matched if c in df_matched.columns]].head(5))

if not df_missing.empty:
    print("\n--- 🔴 Top 5 STILL MISSING CAN Requirements ---")
    display_cols_missing = ['REQ ID', 'Can Signal', 'match_key']
    display(df_missing[[c for c in display_cols_missing if c in df_missing.columns]].head(5))

Loading existing Excel sheet: Intermediate_Requirements.xlsx (Tab: E2E_CAN)...
Saving updated CAN data back to Excel...
✅ CAN Excel update completed successfully!

=== FINAL CAN VALIDATION STATISTICS ===
Total Requirements Processed : 919
✅ Matched (Normalized)       : 918 (99.89%)
❌ Still Missing             : 1

--- 🟢 Top 5 SUCCESSFULLY MATCHED CAN Requirements ---


,REQ ID,Can Signal,Cluster,TX_Node,RX_Nodes,Periodicity_ms,Signal_String,CAPL_Type
0,REQ_E2E_CAN_023,ABSinRegulation,CAN_EXT,PCU_CP_1,AAM,20,CAN_EXT::PCU_CP_1::VDC_HS_1002::IABSinRegulation,int
1,REQ_E2E_CAN_024,ABSMalfunction,CAN_EXT,PCU_CP_1,AAM,20,CAN_EXT::PCU_CP_1::VDC_HS_1002::IABSMalfunction,int
2,REQ_E2E_CAN_025,ABSstateDisplayRequest,CAN_FD_CHASSIS,EBAM,,100,CAN_FD_CHASSIS::EBAM::VDC_HS_1006::IABSstateDisplayRequest,int
3,REQ_E2E_CAN_026,AC_Historical_Fan1_Request,CAN_FD_PCU_LL,PCU_LLCE_2,PCU_CP_1,1000,CAN_FD_PCU_LL::PCU_LLCE_2::PWT_FD_2004::IAC_Historical_Fan1_Request,int
4,REQ_E2E_CAN_027,ACC_BrakingPerformanceStatus,CAN_FD_CHASSIS,EBAM,,20,CAN_FD_CHASSIS::EBAM::VDC_FD_1001::IACC_BrakingPerformanceStatus,int



--- 🔴 Top 5 STILL MISSING CAN Requirements ---


,REQ ID,Can Signal,match_key
286,REQ_E2E_CAN_012,Global_BrakeWheelTorqueRequest_V2,global_brakewheeltorquerequest_v2


In [6]:
# Search for any signal containing "BrakeWheelTorque"
print("🔍 Searching ARXML for partial matches of 'BrakeWheelTorque'...")

deep_search = df_signals[df_signals['Signal_Name'].str.contains('BrakeWheelTorque', case=False, na=False)]

if not deep_search.empty:
    print(f"✅ Found {len(deep_search)} related signals:")
    # We show the unique raw names to see the exact spelling/suffix
    print(deep_search['Signal_Name'].unique().tolist())
else:
    print("❌ No signals containing 'BrakeWheelTorque' exist in the ARXML.")

🔍 Searching ARXML for partial matches of 'BrakeWheelTorque'...
✅ Found 35 related signals:
['IBrakeWheelTorqueEstimation', 'IElecBrakeWheelTorqueApplied', 'IElecBrakeWheelTorqueRequest', 'IGlobal_BrakeWheelTorqueOrder_V2', 'IGlobal_BrakeWheelTorqueReq_V2', 'IRr_ElecBrakeWheelTorqueReq']


In [4]:
def update_can_intermediate_sheet(arxml_path: str, excel_path: str, sheet_name: str = "E2E_CAN"):
    log.info("Starting CAN Database Dump to Intermediate Sheet...")

    # ==========================================
    # 1. INVOKE THE CAN PARSER
    # ==========================================
    parser = CANSignalParser(arxml_path)
    df_signals = parser.to_dataframe()
    
    if df_signals.empty:
        log.error("CAN Parsing failed. Aborting Excel update.")
        return None, None

    # ==========================================
    # 2. LOAD EXISTING EXCEL SHEET
    # ==========================================
    if not os.path.exists(excel_path):
        log.error(f"Excel file '{excel_path}' not found.")
        return None, None

    try:
        df_req = pd.read_excel(excel_path, sheet_name=sheet_name)
    except Exception as e:
        log.error(f"Failed to read sheet '{sheet_name}': {e}")
        return None, None

    # ==========================================
    # 3. NORMALIZE KEYS FOR MATCHING
    # ==========================================
    req_signal_column = "Can Signal" # Ensure this exactly matches your Excel header!
    
    if req_signal_column not in df_req.columns:
        log.error(f"Column '{req_signal_column}' not found. Available columns: {df_req.columns.tolist()}")
        return None, None

    # Requirements: Get the column, strip whitespace, and force to LOWERCASE
    df_req['match_key'] = df_req[req_signal_column].astype(str).str.strip().str.lower()
    
    # Database: Strip whitespace, force to LOWERCASE, and REMOVE the "i_" prefix!
    df_signals['match_key'] = df_signals['Signal_Name'].astype(str).str.strip().str.lower()
    df_signals['match_key'] = df_signals['match_key'].str.replace(r'^i', '', regex=True)

    # Drop duplicates on the match key so we don't multiply rows in the Excel sheet
    df_signals_unique = df_signals.drop_duplicates(subset=['match_key'])

    # Columns we want to pull from the CAN Database
    columns_to_add = [
        'match_key', 'Signal_String', 'Cluster', 'TX_Node', 'RX_Nodes', 
        'Periodicity_ms', 'Base_Type', 'CAPL_Type', 'Unit', 
        'Resolution', 'Offset', 'Min', 'Max'
    ]

    # Clean up old columns to prevent duplication suffixing (like _x, _y)
    existing_cols = df_req.columns.tolist()
    cols_to_drop = [c for c in columns_to_add if c in existing_cols and c != 'match_key']
    df_req = df_req.drop(columns=cols_to_drop)

    # ==========================================
    # 4. PERFORM THE MERGE
    # ==========================================
    df_updated = pd.merge(
        df_req, 
        df_signals_unique[columns_to_add], 
        on='match_key', 
        how='left'
    )

    # Flag for our statistics output
    df_updated['Is_Found_In_DB'] = df_updated['Signal_String'].notna()

    # Split for the visual output before we drop the temp keys
    df_matched = df_updated[df_updated['Is_Found_In_DB'] == True].copy()
    df_missing = df_updated[df_updated['Is_Found_In_DB'] == False].copy()

    # Remove temporary match flags before saving to Excel
    df_updated = df_updated.drop(columns=['match_key', 'Is_Found_In_DB'])

    # ==========================================
    # 5. WRITE BACK TO EXCEL
    # ==========================================
    log.info(f"Saving updated CAN data back to {excel_path} (Tab: {sheet_name})...")
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            df_updated.to_excel(writer, sheet_name=sheet_name, index=False)
        log.info("✅ CAN Excel update completed successfully!")
    except PermissionError:
        log.error("Permission Denied: Close the Excel file and try again.")
    except Exception as e:
        log.error(f"Failed to write to Excel: {e}")

    return df_matched, df_missing

In [ ]:
# Hunt for ABS
abs_matches = df_signals[df_signals['Signal_Name'].str.contains('abs', case=False, na=False)]
if not abs_matches.empty:
    print("\n✅ Found 'ABS' signals in ARXML. Here are the raw names:")
    # We use .unique() to avoid printing the exploded duplicate rows
    for name in abs_matches['Signal_Name'].unique()[:10]:
        print(f"  -> {name}")
else:
    print("\n❌ CRITICAL: No signals containing 'abs' exist in the ARXML.")

# Hunt for Fan
fan_matches = df_signals[df_signals['Signal_Name'].str.contains('fan', case=False, na=False)]
if not fan_matches.empty:
    print("\n✅ Found 'FAN' signals in ARXML. Here are the raw names:")
    for name in fan_matches['Signal_Name'].unique()[:5]:
        print(f"  -> {name}")
else:
    print("\n❌ CRITICAL: No signals containing 'fan' exist in the ARXML.")